### `GroupBy` en Spark

En este cuaderno calculamos los ingresos por hora y zona de recogida para los taxis de Nueva York. Aprovecharemos para entender cómo ejecuta Spark un `GroupBy` internamente, porque el mecanismo que usa tiene consecuencias directas sobre el rendimiento de nuestros trabajos.

In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('groupby-en-spark') \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/07 15:26:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


#### Calcular los ingresos de los taxis verdes

Leemos los datos de los taxis verdes y los registramos como vista temporal con `createOrReplaceTempView`. A partir de aquí podemos referirnos a ellos por nombre en cualquier consulta SQL.

In [2]:
df_green = spark.read.parquet('/data/parquet/green/*/*')
df_green.createOrReplaceTempView('green')

La consulta agrupa los viajes por hora y zona de recogida y calcula dos métricas: la suma de los importes totales y el número de viajes. `date_trunc('hour', ...)` trunca la marca de tiempo al inicio de la hora, de modo que todos los viajes de una misma hora queden en el mismo grupo. El `GROUP BY 1, 2` hace referencia a las dos primeras columnas del `SELECT` por posición. El filtro descarta registros erróneos de fechas anteriores a 2020.

In [3]:
df_green_revenue = spark.sql("""
SELECT
    date_trunc('hour', lpep_pickup_datetime) AS hour,
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    hour, zone
""")

Antes de escribir el resultado llamamos a `repartition(20)`. Tras un `GroupBy`, Spark produce por defecto 200 particiones de salida, lo que en conjuntos de datos pequeños genera 200 ficheros minúsculos. Reducirlos a 20 es más manejable. En la interfaz de Spark este paso de reparticionado aparece como una tercera etapa adicional, ya que también requiere redistribuir los datos.

In [4]:
df_green_revenue \
    .repartition(20) \
    .write.parquet('/data/report/revenue/green', mode='overwrite')

26/03/07 15:27:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/07 15:27:09 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/07 15:27:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                